# Session 13.2 Practice: Prepare Image Batches

Define the training and test DataLoaders, inspect image and label batches, then train
and test a CNN using the selected device. Write your loader definitions in Section 3
before running the remaining cells.


## 1. Prepare the runtime and digit folders

In Google Colab, choose **Runtime → Change runtime type → Hardware accelerator → T4 GPU**
(or another available GPU), then click **Save** and **Connect**.
Upload `digit_images.zip` through the Files sidebar into the working folder.
Run the following commands in a Colab code cell to extract the images and list
the class folders before continuing to Section 2.

On a Mac, open this notebook in a local Jupyter runtime with PyTorch and torchvision.
Extract the ZIP so `sample_data/digit_images/` is in the notebook working folder.
The code selects CUDA when available, otherwise a Mac GPU through MPS, or the CPU.


```python
!unzip -q -o digit_images.zip
!ls sample_data/digit_images
```


## 2. Load and split the images

Prepare single-channel 28 × 28 tensors and randomly split the
240 images into 192 training and 48 test images. The two subsets do not overlap.


In [1]:
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", DEVICE)  # Prefer CUDA, then Mac GPU (MPS), then CPU.

from torchvision import datasets, transforms
from torch.utils.data import random_split

DIGIT_ROOT = "sample_data/digit_images"
IMAGE_SIZE = (28, 28)

digit_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),  # Keep one intensity channel.
    transforms.Resize(IMAGE_SIZE),  # Set the height and width to 28 pixels.
    transforms.ToTensor(),  # Convert pixels to a float tensor with values in [0, 1].
])
full_dataset = datasets.ImageFolder(DIGIT_ROOT, transform=digit_transform)  # Read labels from class folders.
print("Images:", len(full_dataset), "Classes:", full_dataset.classes)

SEED = 42
torch.manual_seed(SEED) 
TRAIN_FRACTION = 0.8  # Reserve 80% of the images for training.
train_size = int(TRAIN_FRACTION * len(full_dataset))  # Calculate the number of training images.
test_size = len(full_dataset) - train_size  # Use the remaining 20% for testing.
# Randomly assign each image to one subset, keeping training and test images separate.
train_dataset, test_dataset = random_split(
    full_dataset, [train_size, test_size],  # Return training and test subsets in this order.
)
print("Training images:", len(train_dataset), "Test images:", len(test_dataset))


Device: mps
Images: 240 Classes: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
Training images: 192 Test images: 48


## 3. Create the DataLoaders

Write the complete definitions of `train_loader` and `test_loader` in the cell below.

- Import DataLoader and define `BATCH_SIZE = 10`.
- Create `train_loader` from `train_dataset`, using `BATCH_SIZE` and shuffling the training order.
- Create `test_loader` from `test_dataset`, using the same batch size and keeping the test order stable.

Use the supplied print statement to check your work: expect 20 training batches and 5 test batches.
The final training and test batches contain two and eight images respectively.


In [ ]:
# Import DataLoader and define BATCH_SIZE as 10.


# Write the complete train_loader definition here.


# Write the complete test_loader definition here.


print("Training batches:", len(train_loader), "Test batches:", len(test_loader))


## 4. Inspect the image and label batches

Retrieve one batch from each loader. Both first image batches should have shape `[10, 1, 28, 28]`
and both label batches should have shape `[10]`. The loaders prepare these tensors on the CPU;
move them to `DEVICE` before using them with the CNN.


In [ ]:
images, labels = next(iter(train_loader))  # Retrieve one training image batch and its labels.
test_images, test_labels = next(iter(test_loader))  # Retrieve one test image batch and its labels.

images, labels = images.to(DEVICE), labels.to(DEVICE)  # Move the training input and target tensors to the selected device.
test_images, test_labels = test_images.to(DEVICE), test_labels.to(DEVICE)  # Move the test tensors to the same device.
print("Training shapes:", tuple(images.shape), tuple(labels.shape))
print("Test shapes:", tuple(test_images.shape), tuple(test_labels.shape))
print("Input device:", images.device, "Target device:", labels.device)


## 5. Train and test the CNN

Use your DataLoaders to train on all training batches, then test on the separate test images.
The supplied code runs on `DEVICE` and displays ten test images with true and predicted labels.


### 5.1 Create the model

The CNN accepts `[N, 1, 28, 28]` image tensors and returns `[N, 10]` class logits.
Keep the model and input tensors on the same device.


In [ ]:
from torch import nn

torch.manual_seed(SEED)  # Reproduce the initial model weights.
NUMBER_OF_CLASSES = len(full_dataset.classes)
POOLED_SIZE = IMAGE_SIZE[0] // 2

model = nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=3, padding=1),  # Extract eight feature maps, each 28 × 28.
    nn.ReLU(),  # Apply a non-linear activation.
    nn.MaxPool2d(kernel_size=2),  # Reduce each feature map to 14 × 14.
    nn.Flatten(start_dim=1),  # Flatten features while keeping the batch dimension.
    nn.Linear(8 * POOLED_SIZE * POOLED_SIZE, NUMBER_OF_CLASSES),  # Return ten class logits.
).to(DEVICE)  # Move the model weights to the same device as the input tensor.
LEARNING_RATE = 0.001
loss_function = nn.CrossEntropyLoss()  # Compare raw class logits with integer labels.
optimiser = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)  # Set the learning rate for weight updates.
print(model)


### 5.2 Train on the image batches

Train for 30 epochs. Each epoch visits all 20 training batches and updates the weights
once per batch. Report the mean loss across the 192 training images every ten epochs.


In [ ]:
EPOCHS = 30
REPORT_EVERY = 10

for epoch in range(EPOCHS):
    model.train()  # Select training behaviour.
    total_loss, training_images_seen = 0.0, 0
    for train_images, train_labels in train_loader:  # Visit every training batch in this epoch.
        train_images, train_labels = train_images.to(DEVICE), train_labels.to(DEVICE)  # Move inputs and targets to the model device.
        optimiser.zero_grad()  # Clear gradients from the previous batch.
        logits = model(train_images)  # Predict ten class logits for each image in this batch.
        loss = loss_function(logits, train_labels)  # Calculate the mean loss for this batch.
        loss.backward()  # Calculate parameter gradients.
        optimiser.step()  # Update weights using this batch.
        total_loss += loss.item() * train_labels.size(0)  # Weight batch loss by its actual image count.
        training_images_seen += train_labels.size(0)
    if (epoch + 1) % REPORT_EVERY == 0:
        print(f"Epoch {epoch + 1}/{EPOCHS}: training loss={total_loss / training_images_seen:.4f}")


### 5.3 Test and display predictions

Process all 5 test batches without updating weights. Combine the results to calculate
accuracy across all 48 test images and display ten predictions.
Move results to the CPU for plotting. The combined test output should have shape `[48, 10]`.


In [ ]:
import matplotlib.pyplot as plt

model.eval()  # Select evaluation behaviour.
test_image_batches, test_label_batches, test_logit_batches = [], [], []
with torch.no_grad():  # Test without recording gradients or updating weights.
    for test_images, test_labels in test_loader:  # Visit every test batch.
        test_images = test_images.to(DEVICE)  # Put the input batch on the same device as the CNN.
        test_logits = model(test_images)  # Predict class logits for this test batch.
        test_image_batches.append(test_images.cpu())  # Keep images on the CPU for plotting.
        test_label_batches.append(test_labels.cpu())  # Keep the matching true labels.
        test_logit_batches.append(test_logits.cpu())  # Collect outputs from every batch.

test_images = torch.cat(test_image_batches, dim=0)  # Combine batches along the image-count dimension.
test_labels = torch.cat(test_label_batches, dim=0)
test_logits = torch.cat(test_logit_batches, dim=0)
predictions = test_logits.argmax(dim=1)  # Select the highest-scoring class for each image.
correct = (predictions == test_labels).sum().item()
print(f"Test accuracy: {correct / len(test_labels):.1%} ({correct}/{len(test_labels)})")
print("Test input:", tuple(test_images.shape), "Test output:", tuple(test_logits.shape))

class_names = full_dataset.classes
NUMBER_OF_IMAGES = 10
figure, axes = plt.subplots(2, 5, figsize=(14, 5))  # Show ten test images in two rows.
for position, axis in zip(range(NUMBER_OF_IMAGES), axes.flat):
    actual = int(test_labels[position].item())
    predicted = int(predictions[position].item())
    axis.imshow(test_images[position].squeeze(0), cmap="gray", vmin=0, vmax=1)  # Display one greyscale test image.
    axis.set_title(f"True: {class_names[actual]} | Pred: {class_names[predicted]}")
    axis.axis("off")
plt.tight_layout()
plt.show()
